# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

I'm reviewing these constructively, the way I'd want my own notebook reviewed, not to grade the paper.

In [1]:
findings_reviewed = [
    {
        "finding": "Health Score model: Random Forest R^2 = 0.84 using position, impressions, scroll depth and CTR",
        "where_label_comes_from": "The paper's own health score formula is impressions(30) + position(30) + CTR(20) + scroll_depth(20). The model's target IS that formula.",
        "methodology_question": "If the label is a weighted sum of exactly those four inputs, doesn't a model trained to predict it have to rediscover the formula? What would R^2 look like on an outcome the formula did not construct, such as next-quarter organic revenue?",
    },
    {
        "finding": "Refreshed content (271+ days old, updated) gets ~57x more impressions than non-refreshed content of the same age",
        "where_label_comes_from": "Impressions_90d, compared across two groups of existing pages (refreshed vs not), not a controlled before/after on the same pages.",
        "methodology_question": "Could editors be refreshing the pages that already show promise (more search volume, more backlinks) rather than refreshing causing the lift? Is there a before/after view of the SAME pages' impressions, pre- and post-refresh?",
    },
]
findings_reviewed

[{'finding': 'Health Score model: Random Forest R^2 = 0.84 using position, impressions, scroll depth and CTR',
  'where_label_comes_from': "The paper's own health score formula is impressions(30) + position(30) + CTR(20) + scroll_depth(20). The model's target IS that formula.",
  'methodology_question': "If the label is a weighted sum of exactly those four inputs, doesn't a model trained to predict it have to rediscover the formula? What would R^2 look like on an outcome the formula did not construct, such as next-quarter organic revenue?"},
 {'finding': 'Refreshed content (271+ days old, updated) gets ~57x more impressions than non-refreshed content of the same age',
  'where_label_comes_from': 'Impressions_90d, compared across two groups of existing pages (refreshed vs not), not a controlled before/after on the same pages.',
  'methodology_question': "Could editors be refreshing the pages that already show promise (more search volume, more backlinks) rather than refreshing causing th

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Setup.** Same data, same label (`is_declining`), same three model families as Week 5 (Logistic Regression, Random Forest, Gradient Boosting), same metrics. Only the split design changes between the two tables below.

In [2]:
import json, os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

for c in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df["log_" + c] = np.log1p(df[c])

NUM = ["search_volume", "competition", "cpc", "word_count", "char_count",
       "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
       "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
       "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"]
CAT = ["competition_level", "content_type", "main_intent"]
X = df[NUM + CAT]
y = df["is_declining"].values
groups = df["client_id"].values

def preprocess(scale):
    num = make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()) if scale else SimpleImputer(strategy="median", add_indicator=True)
    cat = make_pipeline(SimpleImputer(strategy="constant", fill_value="missing"), OneHotEncoder(handle_unknown="ignore"))
    return ColumnTransformer([("num", num, NUM), ("cat", cat, CAT)])

def models():
    return {
        "Logistic Regression": make_pipeline(preprocess(True), LogisticRegression(max_iter=3000)),
        "Random Forest":       make_pipeline(preprocess(False), RandomForestClassifier(300, max_depth=10, min_samples_leaf=20, random_state=SEED, n_jobs=-1)),
        "Gradient Boosting":   make_pipeline(preprocess(False), HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=200, random_state=SEED)),
    }

def precision_at_k(y_true, score, k):
    order = np.lexsort((np.arange(len(score)), -score))[:k]
    return y_true[order].mean()

def cross_val(splits, model_fn, X_):
    out = []
    for tr, te in splits:
        m = model_fn().fit(X_.iloc[tr], y[tr])
        s = m.predict_proba(X_.iloc[te])[:, 1]
        out.append((roc_auc_score(y[te], s), precision_at_k(y[te], s, 100), precision_at_k(y[te], s, 500)))
    a = np.array(out)
    return a[:, 0].mean(), a[:, 0].std(), a[:, 1].mean(), a[:, 2].mean()

print(f"base rate (random picking): {y.mean():.3f}")

base rate (random picking): 0.542


**BEFORE: a naive 5-fold split, stratified on the label, ignoring `client_id`.** This is the split most people reach for by default, and the one Week 5 deliberately avoided.

In [3]:
# BEFORE: a naive split that ignores client_id. Same model family as my Week-5 notebook.
naive_splits = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(X, y))

before_rows = []
for name, fn in models().items():
    auc_mean, auc_std, p100, p500 = cross_val(naive_splits, lambda n=name: models()[n], X)
    before_rows.append({"method": name, "ROC AUC": f"{auc_mean:.3f} +/- {auc_std:.3f}", "precision@100": round(p100, 3), "precision@500": round(p500, 3)})
before = pd.DataFrame(before_rows)
before

,method,ROC AUC,precision@100,precision@500
0,Logistic Regression,0.709 +/- 0.006,0.858,0.824
1,Random Forest,0.764 +/- 0.005,0.940,0.897
2,Gradient Boosting,0.772 +/- 0.003,0.934,0.909


**AFTER: GroupKFold by `client_id`, same models, same features.** An assertion in the code confirms no client appears on both sides of any fold.

In [4]:
# AFTER: the same models, same features, same metrics -- only the split changes to GroupKFold by client_id.
grouped_splits = list(GroupKFold(n_splits=5).split(X, y, groups))
for tr, te in grouped_splits:
    assert not (set(groups[tr]) & set(groups[te])), "a client leaked across the split"

after_rows = []
for name in models():
    auc_mean, auc_std, p100, p500 = cross_val(grouped_splits, lambda n=name: models()[n], X)
    after_rows.append({"method": name, "ROC AUC": f"{auc_mean:.3f} +/- {auc_std:.3f}", "precision@100": round(p100, 3), "precision@500": round(p500, 3)})
after = pd.DataFrame(after_rows)
after

,method,ROC AUC,precision@100,precision@500
0,Logistic Regression,0.658 +/- 0.030,0.800,0.747
1,Random Forest,0.668 +/- 0.036,0.682,0.736
2,Gradient Boosting,0.677 +/- 0.042,0.776,0.738


**The gap.** Every model scores lower under the honest split, and the drop is not small:

| Method | Naive split (before) | Grouped split (after) | Drop |
|---|---|---|---|
| Logistic Regression | 0.709 | 0.658 | -0.051 |
| Random Forest | 0.764 | 0.668 | -0.096 |
| Gradient Boosting | 0.772 | 0.677 | -0.095 |

The more flexible models (Random Forest, Gradient Boosting) look best under the naive split and lose the most once client memorization is removed. Under the naive split they also beat Logistic Regression by about 0.06, but under the grouped split the gap shrinks to about 0.01 to 0.02, inside the fold-to-fold noise. That matches Week 5's choice of Logistic Regression: the naive split would have pointed to the wrong model.

In [5]:
compare = before[["method", "ROC AUC"]].merge(after[["method", "ROC AUC"]], on="method", suffixes=(" (naive, before)", " (grouped, after)"))
compare

,method,"ROC AUC (naive, before)","ROC AUC (grouped, after)"
0,Logistic Regression,0.709 +/- 0.006,0.658 +/- 0.030
1,Random Forest,0.764 +/- 0.005,0.668 +/- 0.036
2,Gradient Boosting,0.772 +/- 0.003,0.677 +/- 0.042


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I run the same checklist I used on the paper, on my own final feature set from Week 5.

In [6]:
checklist = {
    "Timeline drawn: features strictly before the label window":          "FAIL (see below) -- the current feature set overlaps the label window",
    "No label-derived or sibling columns in the features":                "PASS -- is_declining, trend_direction, trend_pct, *_last_30d, *_prev_30d are all excluded",
    "No product flags / existing-system scores as features":              "PASS -- the Week-4 rule score is used only as the baseline, never as an input",
    "Split grouped by the repeating entity (client_id)":                  "PASS from this point on -- see section 2",
    "Base rate printed next to every metric":                             "PASS -- printed above",
    "Top feature importance sanity-checked":                              "see below",
    "Metrics recomputed out-of-fold, never in-sample":                    "PASS -- all AUC/precision above are from held-out folds",
}
pd.Series(checklist).to_frame("status")

,status
Timeline drawn: features strictly before the label window,FAIL (see below) -- the current feature set ov...
No label-derived or sibling columns in the features,"PASS -- is_declining, trend_direction, trend_p..."
No product flags / existing-system scores as features,PASS -- the Week-4 rule score is used only as ...
Split grouped by the repeating entity (client_id),PASS from this point on -- see section 2
Base rate printed next to every metric,PASS -- printed above
Top feature importance sanity-checked,see below
"Metrics recomputed out-of-fold, never in-sample",PASS -- all AUC/precision above are from held-...


**Drawing the timeline.** The data dictionary defines `*_90d` as the last 30 days plus the previous 30 days plus one earlier 30-day block. `trend_direction` (and so `is_declining`) is built from `impressions_last_30d` vs `impressions_prev_30d`. So every `*_90d` feature I used in Week 5 **contains** the two windows the label is computed from. This fails the timeline check: the features are not fully knowable before the label window closes.

In [7]:
# The label's own windows:
#   impressions_last_30d, impressions_prev_30d  -- these build trend_direction / trend_pct / is_declining
# My current features use *_90d, which the data dictionary defines as last_30d + prev_30d + an earlier 30 days.
# So impressions_90d CONTAINS impressions_last_30d: the feature is not knowable before the label window closes.

for base in ["impressions", "clicks", "sessions"]:
    total_90d  = df[f"{base}_90d"]
    last_30d   = df[f"{base}_last_30d"]
    prev_30d   = df[f"{base}_prev_30d"]
    older_30d  = total_90d - last_30d - prev_30d
    print(f"{base}: rows where 90d < last30d+prev30d (should be 0): {int((older_30d < 0).sum())}  | "
          f"median older_30d: {older_30d.median():.0f}")

impressions: rows where 90d < last30d+prev30d (should be 0): 0  | median older_30d: 308
clicks: rows where 90d < last30d+prev30d (should be 0): 0  | median older_30d: 0
sessions: rows where 90d < last30d+prev30d (should be 0): 0  | median older_30d: 1


In [8]:
# The deliberate-leak harness from Week 3, re-run here on the final feature set, to prove the test itself works.
df["LEAK"] = df["impressions_last_30d"] / (df["impressions_prev_30d"] + 1)   # directly built from the label's own windows

NUM_WITH_LEAK = NUM + ["LEAK"]   # the ColumnTransformer must be told about the new column, or it silently drops it
def preprocess_leak(scale):
    num = make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()) if scale else SimpleImputer(strategy="median", add_indicator=True)
    cat = make_pipeline(SimpleImputer(strategy="constant", fill_value="missing"), OneHotEncoder(handle_unknown="ignore"))
    return ColumnTransformer([("num", num, NUM_WITH_LEAK), ("cat", cat, CAT)])

leak_auc, leak_std, leak_p100, _ = cross_val(grouped_splits, lambda: make_pipeline(preprocess_leak(True), LogisticRegression(max_iter=3000)), df[NUM_WITH_LEAK + CAT])
base_auc = float(after.loc[after["method"] == "Logistic Regression", "ROC AUC"].iloc[0].split(" ")[0])
print(f"Logistic Regression, grouped split, WITHOUT LEAK: {base_auc:.3f}")
print(f"Logistic Regression, grouped split, WITH LEAK:    {leak_auc:.3f} +/- {leak_std:.3f}   (precision@100 = {leak_p100:.3f})")
df = df.drop(columns=["LEAK"])

Logistic Regression, grouped split, WITHOUT LEAK: 0.658
Logistic Regression, grouped split, WITH LEAK:    0.716 +/- 0.100   (precision@100 = 0.876)


**The fix.** I rebuild `impressions_90d`, `clicks_90d` and `sessions_90d` as "90d minus last_30d minus prev_30d", which is the one 30-day block that sits *before* both of the label's windows, then refit on that smaller, honestly-timed feature set (`PRE_LABEL_NUM`). I drop `avg_position`, `ctr`, `engagement_rate`, `scroll_rate` and `ai_traffic_pct`: the data dictionary gives no way to split these into a pre-label portion, so I can't clear them and don't use them here.

In [9]:
# Rebuild every 90-day column as "the 30 days BEFORE the label window" only, so it cannot see the label's own windows.
for base in ["impressions", "clicks", "sessions"]:
    df[f"{base}_old30"] = (df[f"{base}_90d"] - df[f"{base}_last_30d"] - df[f"{base}_prev_30d"]).clip(lower=0)
    df[f"log_{base}_old30"] = np.log1p(df[f"{base}_old30"])

df["ctr_old30"] = np.where(df["impressions_old30"] > 0, df["clicks_old30"] / df["impressions_old30"] * 100, np.nan)

PRE_LABEL_NUM = ["search_volume", "competition", "cpc", "word_count", "char_count", "content_age_days",
                  "days_since_last_update", "log_impressions_old30", "log_clicks_old30", "log_sessions_old30", "ctr_old30"]
# avg_position, ctr, engagement_rate, scroll_rate, ai_traffic_pct, days_with_impressions/sessions are dropped:
# the data dictionary does not separate their last-30/prev-30 portions from the 90-day figure, so I cannot clear them.

X_pre = df[PRE_LABEL_NUM + CAT]

def preprocess_pre(scale):
    num = make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()) if scale else SimpleImputer(strategy="median", add_indicator=True)
    cat = make_pipeline(SimpleImputer(strategy="constant", fill_value="missing"), OneHotEncoder(handle_unknown="ignore"))
    return ColumnTransformer([("num", num, PRE_LABEL_NUM), ("cat", cat, CAT)])

def models_pre():
    return {
        "Logistic Regression (pre-label features)": make_pipeline(preprocess_pre(True), LogisticRegression(max_iter=3000)),
        "Gradient Boosting (pre-label features)":    make_pipeline(preprocess_pre(False), HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=200, random_state=SEED)),
    }

pre_rows = []
for name in models_pre():
    out = []
    for tr, te in grouped_splits:
        m = models_pre()[name].fit(X_pre.iloc[tr], y[tr])
        s = m.predict_proba(X_pre.iloc[te])[:, 1]
        out.append((roc_auc_score(y[te], s), precision_at_k(y[te], s, 100), precision_at_k(y[te], s, 500)))
    a = np.array(out)
    pre_rows.append({"method": name, "ROC AUC": f"{a[:,0].mean():.3f} +/- {a[:,0].std():.3f}", "precision@100": round(a[:,1].mean(),3), "precision@500": round(a[:,2].mean(),3)})
pd.DataFrame(pre_rows)

,method,ROC AUC,precision@100,precision@500
0,Logistic Regression (pre-label features),0.660 +/- 0.060,0.782,0.757
1,Gradient Boosting (pre-label features),0.626 +/- 0.040,0.644,0.671


**What the leak harness and the fix show, side by side:**

| Version | ROC AUC (grouped split) |
|---|---|
| Week-5 features, no leak column | 0.658 |
| Week-5 features **+ deliberate LEAK column** | 0.716 (+0.058) |
| Honest pre-label-only features, no 90d overlap | 0.660 |

Two things follow. First, the harness works: adding a column built straight from the label's own windows pushes AUC up by about 0.06, confirming the test can catch a real leak (not as large a jump as a near-1.0 case, because this ratio is noisy and the label itself is thresholded, but a real and repeatable jump). Second, and this is the one I didn't expect: once I remove the overlapping `*_90d` windows entirely and keep only the one clean 30-day block before the label, the honest model (0.660) scores about the same as the original Week-5 model (0.658). So the overlap was not secretly propping up the Week-5 number. Why: the signal mostly lives in the ratio between recent clicks and recent impressions (a CTR-like pattern), and that pattern is nearly as visible in the earlier 30-day block as in the full 90 days. I would not have guessed this without running the test; it also means the Week-5 model's honest-looking score doesn't need revising, but the feature *definitions* still should be cleaned up, because the same overlap could matter on a different label or dataset.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Before (Week 5, too strong):** "A simple Logistic Regression ranks the full list much better than my Week-4 rule (AUC 0.66 vs 0.53) and matches it at the very top."

**After, in safe language:**

> *Under a client-grouped validation split, Logistic Regression was **observed** to order this one month's content-refresh data by decline risk with an AUC of 0.658, compared with 0.526 for the Week-4 rule on the same folds. At the top of the ranked list (the top 100 and top 500 pages), the two were **measured** to perform similarly. The direction of this result held even after rebuilding the features to remove a window overlap with the label, which is some evidence it isn't an artifact of that overlap. This is **decision-support** for which pages an editor might review first, not a forecast: the label is a same-month proxy, not a future outcome, and the pattern has only been checked on 32 clients in one snapshot month. I would call the improvement over the Week-4 rule **directional** until it's checked against the sealed final month.*

What changed: "ranks much better" became a specific, attributed number with its comparison point named. "Matches it at the very top" became "measured to perform similarly," because a 2-point gap in one month is not a proven match. I added the grouped-split and leak-audit caveats, and replaced the implicit claim of generality with "one month" and "32 clients."

In [10]:
metrics = {
    "task": "w06_validation_audit",
    "split_before_naive": {m: v for m, v in zip(before["method"], before["ROC AUC"])},
    "split_after_grouped": {m: v for m, v in zip(after["method"], after["ROC AUC"])},
    "leak_harness": {"without_leak": round(base_auc, 3), "with_leak": round(float(leak_auc), 3)},
    "pre_label_features_grouped": {r["method"]: r["ROC AUC"] for r in pre_rows},
}
os.makedirs("../outputs", exist_ok=True)
with open("../outputs/w06_validation_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))

{
  "task": "w06_validation_audit",
  "split_before_naive": {
    "Logistic Regression": "0.709 +/- 0.006",
    "Random Forest": "0.764 +/- 0.005",
    "Gradient Boosting": "0.772 +/- 0.003"
  },
  "split_after_grouped": {
    "Logistic Regression": "0.658 +/- 0.030",
    "Random Forest": "0.668 +/- 0.036",
    "Gradient Boosting": "0.677 +/- 0.042"
  },
  "leak_harness": {
    "without_leak": 0.658,
    "with_leak": 0.716
  },
  "pre_label_features_grouped": {
    "Logistic Regression (pre-label features)": "0.660 +/- 0.060",
    "Gradient Boosting (pre-label features)": "0.626 +/- 0.040"
  }
}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.